# ARLSTM Grid Sweep Multi-Model Evaluation

This notebook evaluates and compares **all 27 trained ARLSTM grid sweep models** stored locally in `model-runs/arlstm_grid_sweep/`.

### Hyperparameter Grid Axes Evaluated:
- **Learning Rate (`lr`)**: `0.0001`, `0.001`, `0.01`
- **Masking Fraction (`mf`)**: `0.1` (10%), `0.9` (90%), `0.99` (99%)
- **Hidden Size (`hs`)**: `128`, `256`, `512`
- **Random Seed (`seed`)**: `42`


In [ ]:
%matplotlib inline
import os
import sys
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xarray as xr
from IPython.display import display, HTML

# Resolve repository root
cwd = Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent, Path("/usr/local/google/home/kruparell/flood-forecasting")] if (p / "googlehydrology").is_dir() or (p / "tutorial").is_dir()),
    cwd
)
tut_dir = repo_root / "tutorial" if (repo_root / "tutorial").is_dir() else repo_root

for p in [str(repo_root), str(tut_dir), str(tut_dir / "scripts"), str(tut_dir / "notebooks")]:
    if p not in sys.path:
        sys.path.insert(0, p)

from googlehydrology.utils.config import Config
from googlehydrology.modelzoo.arlstm import ARLSTM
from googlehydrology.evaluation.tester import RegressionTester
from googlehydrology.datasetzoo.multimet import _convert_to_tensor
from googlehydrology.evaluation.metrics import calculate_metrics

def compute_hydro_metrics(obs_np, sim_np):
    obs_da = xr.DataArray(obs_np, dims=["date"])
    sim_da = xr.DataArray(sim_np, dims=["date"])
    return calculate_metrics(obs_da, sim_da, metrics=["NSE", "KGE", "Pearson-r", "RMSE"])

plt.rcParams["font.size"] = 11
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment and GoogleHydrology modules initialized successfully.")


## 1. Multi-Model Grid Sweep Evaluation Loop

We iterate through all local model directories in `model-runs/arlstm_grid_sweep/`, load configurations, scalers, and PyTorch checkpoints, run inference for both **Full Streamflow Input** and **Zero Streamflow Input**, and aggregate performance metrics.


In [ ]:
t0 = time.time()
grid_dir = repo_root / "model-runs" / "arlstm_grid_sweep"
model_folders = sorted([d for d in grid_dir.iterdir() if d.is_dir()])
print(f"Evaluating {len(model_folders)} models in {grid_dir.name}...")

sweep_records = []
detailed_results = {}

for m_dir in model_folders:
    m_name = m_dir.name
    match = re.match(r"lr([\d\.]+)_mf([\d\.]+)_hs(\d+)_seed(\d+)", m_name)
    lr = float(match.group(1)) if match else None
    mf = float(match.group(2)) if match else None
    hs = int(match.group(3)) if match else None

    ar_cfg = Config(m_dir / "config.yml")
    ar_cfg._cfg["run_dir"] = m_dir
    ar_cfg._cfg["base_run_dir"] = m_dir
    if not Path(ar_cfg.statics_data_dir).exists():
        ar_cfg._cfg["statics_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_V2")
    if not Path(ar_cfg.dynamics_data_dir).exists():
        ar_cfg._cfg["dynamics_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_MultiMet")
    if not Path(ar_cfg.targets_data_dir).exists():
        ar_cfg._cfg["targets_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_V2")

    if (m_dir / "sampled_task0_200basins_seed42.txt").exists():
        ar_cfg._cfg["test_basin_file"] = m_dir / "sampled_task0_200basins_seed42.txt"
        ar_cfg._cfg["train_basin_file"] = m_dir / "sampled_task0_200basins_seed42.txt"
    elif not Path(ar_cfg.test_basin_file).exists():
        ar_cfg._cfg["test_basin_file"] = tut_dir / "basin-lists" / "10-basin-test.txt"
        ar_cfg._cfg["train_basin_file"] = tut_dir / "basin-lists" / "10-basin-train.txt"

    scaler_path = m_dir / "scaler.nc"
    if not scaler_path.exists():
        continue
    scaler_ds = xr.open_dataset(scaler_path)
    mu_q = float(scaler_ds["streamflow_sim"].sel(parameter="mean").values) if "streamflow_sim" in scaler_ds else 1.7772
    sigma_q = float(scaler_ds["streamflow_sim"].sel(parameter="std").values) if "streamflow_sim" in scaler_ds else 3.3810

    pt_files = sorted(list(m_dir.glob("model_epoch*.pt")))
    if not pt_files:
        continue
    latest_ckpt = pt_files[-1]

    tester = RegressionTester(cfg=ar_cfg, run_dir=m_dir, period="test", init_model=True)
    tester.model.eval()

    zarr_path = m_dir / "test" / latest_ckpt.stem / "test_results.zarr"
    if not zarr_path.exists():
        print(f"Executing live evaluation for {m_name}...")
        tester.evaluate()
        zarr_path = sorted(list((m_dir / "test").glob("*/test_results.zarr")))[-1] if (m_dir / "test").exists() else None

    if zarr_path and zarr_path.exists():
        ds_zarr = xr.open_zarr(zarr_path, consolidated=False).compute()
        basins = [str(b) for b in ds_zarr["basin"].values]
        dates = pd.to_datetime(ds_zarr["date"].values)

        nse_full_list, nse_zero_list = [], []
        kge_full_list, kge_zero_list = [], []
        for b in basins:
            obs = ds_zarr["streamflow_obs"].sel(basin=b).isel(freq=0, time_step=0).values.flatten()
            sim_full = np.maximum(0.0, ds_zarr["streamflow_sim"].sel(basin=b).isel(freq=0, time_step=0).values.flatten())

            zero_val = (0.0 - mu_q) / sigma_q
            if b in tester.basins:
                idx = tester.basins.index(b)
                sample = tester.dataset[idx]
                batch_zero = tester.dataset.collate_fn([{k: _convert_to_tensor(k, v) for k, v in sample.items()}])
                if "x_d_hindcast" in batch_zero and "streamflow_shift1" in batch_zero["x_d_hindcast"]:
                    batch_zero["x_d_hindcast"]["streamflow_shift1"] = torch.full_like(batch_zero["x_d_hindcast"]["streamflow_shift1"], zero_val)
                elif "x_d" in batch_zero and "streamflow_shift1" in batch_zero["x_d"]:
                    batch_zero["x_d"]["streamflow_shift1"] = torch.full_like(batch_zero["x_d"]["streamflow_shift1"], zero_val)

                batch_zero = tester.model.pre_model_hook(batch_zero, is_train=False)
                with torch.no_grad():
                    out_zero = tester.model(batch_zero)
                    y_hat_zero = out_zero["y_hat"][0, :, 0].cpu().numpy()
                sim_zero_raw = np.maximum(0.0, y_hat_zero * sigma_q + mu_q)
                if len(sim_zero_raw) < len(obs):
                    sim_zero = np.full_like(obs, np.nan)
                    sim_zero[-len(sim_zero_raw):] = sim_zero_raw
                else:
                    sim_zero = sim_zero_raw[:len(obs)]
            else:
                sim_zero = np.full_like(obs, np.nan)

            m_full = compute_hydro_metrics(obs, sim_full)
            m_zero = compute_hydro_metrics(obs, sim_zero)

            nse_full_list.append(m_full["NSE"])
            nse_zero_list.append(m_zero["NSE"])
            kge_full_list.append(m_full["KGE"])
            kge_zero_list.append(m_zero["KGE"])

        sweep_records.append({
            "Model": m_name,
            "Learning Rate": lr,
            "Masking Fraction": mf,
            "Hidden Size": hs,
            "Median NSE (Full)": np.median(nse_full_list),
            "Mean NSE (Full)": np.mean(nse_full_list),
            "Median KGE (Full)": np.median(kge_full_list),
            "Median NSE (Zero)": np.median(nse_zero_list),
            "Mean NSE (Zero)": np.mean(nse_zero_list),
            "Median KGE (Zero)": np.median(kge_zero_list),
            "NSE Degradation (Full-Zero)": np.median(nse_full_list) - np.median(nse_zero_list),
        })
        detailed_results[m_name] = {"nse_full": nse_full_list, "nse_zero": nse_zero_list}

df_sweep = pd.DataFrame(sweep_records).sort_values(by="Median NSE (Full)", ascending=False)
elapsed = time.time() - t0
print(f"Evaluated {len(df_sweep)} models in {elapsed:.2f}s.")
display(df_sweep.round(4))

## 2. Hyperparameter Performance Heatmaps

Visualizing **Median NSE (Full Input)** across Learning Rate ($Y$-axis) and Masking Fraction ($X$-axis), faceted by Hidden Size.


In [ ]:
hidden_sizes = sorted(df_sweep["Hidden Size"].unique())
fig, axes = plt.subplots(1, len(hidden_sizes), figsize=(18, 5), sharey=True)
fig.patch.set_facecolor("white")

for i, hs_val in enumerate(hidden_sizes):
    sub_df = df_sweep[df_sweep["Hidden Size"] == hs_val]
    pivot = sub_df.pivot(index="Learning Rate", columns="Masking Fraction", values="Median NSE (Full)")
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i], cbar=(i == len(hidden_sizes)-1), vmin=0.0, vmax=1.0)
    axes[i].set_title(f"Hidden Size = {hs_val}", fontweight="bold")
    axes[i].set_xlabel("Masking Fraction", fontweight="bold")
    if i == 0:
        axes[i].set_ylabel("Learning Rate", fontweight="bold")

plt.suptitle("ARLSTM Grid Sweep: Median NSE (Full River Discharge Input)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Robustness Analysis: Full vs. Zero Discharge Input

Analyzing how masking fraction during training impacts model reliance on autoregressive streamflow inputs during inference.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")

sns.barplot(data=df_sweep, x="Masking Fraction", y="NSE Degradation (Full-Zero)", hue="Hidden Size", palette="viridis", ax=ax)
ax.set_title("NSE Performance Degradation when Autoregressive Input is Zeroed Out", fontweight="bold", fontsize=13)
ax.set_xlabel("Training Masking Fraction", fontweight="bold")
ax.set_ylabel("NSE Drop (Median NSE Full - Median NSE Zero)", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Top Performing Models Summary


In [ ]:
print("=== TOP 5 ARLSTM MODELS (RANKED BY MEDIAN NSE FULL INPUT) ===")
display(df_sweep.head(5)[["Model", "Learning Rate", "Masking Fraction", "Hidden Size", "Median NSE (Full)", "Median NSE (Zero)", "NSE Degradation (Full-Zero)"]].round(4))
